In [ ]:
# Installation for Modal Notebooks
%uv pip install gurobipy numpy huggingface-hub pandas pyyaml scikit-learn scipy torch torch-geometric tqdm vector-quantize-pytorch
%uv pip install pacmap

# Imports

In [1]:
import os
import sys

# CRITICAL: Disable all OpenMP threading BEFORE importing any heavy libraries
# Set these FIRST to prevent kernel crashes from nested parallelism
os.environ['OMP_NESTED'] = 'FALSE'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OMP_MAX_ACTIVE_LEVELS'] = '1'  # Use newer API instead of OMP_NESTED
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

# Now safe to import heavy libraries
import numpy as np
import matplotlib.pyplot as plt
import pickle
import pickle as pkl
from sklearn.decomposition import PCA
import pacmap

# Finally import forge (which uses heavy libraries internally)
from forge.embeddings import Forge
from forge.pipeline import pretrain, finetune_integral_gap, mip_to_embeddings, mip_to_gapinfo, mip_to_mipinfo
from forge.utils import Constants

OMP: Info #268: OMP_NESTED variable deprecated, please use OMP_MAX_ACTIVE_LEVELS instead.
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


# Generate Embeddings From Pre-Trained Model

In [ ]:
# Forge model with its pre-trained configuration
forge = Forge(train_config_yaml="../forge/configs/train_config.yaml")

#config = "satlib"
config = "iclr_test_clusters"
#trained_model="iclr_pretrain_clusters_with_relaxed_trained"
trained_model="iclr_forge_pretrain_trained"
input_mips="../data/instances/"
#input_mips="../data/sat_instances/"
# mip_to_embeddings_dict = mip_to_embeddings(forge=forge,
#                                            input_mips=input_mips,
#                                            input_mip_instances_file=f"../data/configs/{config}.txt",
#                                            input_forge_pkl=f"../models/{trained_model}.pkl",
#                                            model_type=Constants.FORGE_PRE_TRAIN,
#                                            output_mip_to_embeddings_pkl=f"{config}_mip_to_embeddings.pkl",
#                                            instance_embedding_only=True)

# Integrate Pre-Extracted SAT Features (Optional)

In [ ]:
# Use SAT Features with FORGE
# Load pre-extracted SAT features and use them as node features for FORGE embedding generation
# SAT constraint features (17-dim) -> use first 4 dims, pad to 10
# SAT variable features (6-dim) -> pad to 10
# Edges: from LP file (actual) or sampled (fallback)

import torch
import gc

config = "satlib"
features_dir = "../data/sat_instances/"
lp_dir = "../data/sat_instances/"
config_file = f"../data/configs/{config}.txt"
output_file = "satlib_mip_to_embeddings_from_sat_features.pkl"

# Read instance names from config file
with open(config_file, 'r') as f:
    instance_names = [line.strip() for line in f if line.strip()]

print(f"Processing {len(instance_names)} instances with SAT features...\n")

sat_embeddings_dict = {}
success_count = 0
device = next(forge.parameters()).device

for idx, instance_name in enumerate(instance_names):
    print(f"[{idx+1}/{len(instance_names)}] {instance_name}...", end=" ", flush=True)
    
    base_name = instance_name.rsplit('.', 1)[0]
    features_file = os.path.join(features_dir, base_name + ".npz")
    lp_file = os.path.join(lp_dir, base_name + ".lp")
    
    if not os.path.exists(features_file):
        print("✗ no features")
        continue
    
    try:
        # Load SAT features
        data = np.load(features_file)
        constraint_features = data['constraint_features']
        variable_features = data['variable_features']
        num_cons = int(data['num_constraints'][0])
        num_vars = int(data['num_vars'][0])
        
        # Extract first 4 constraint features, pad to 10-dim
        constraint_features_4d = constraint_features[:, :4]
        constraint_features_10d = np.hstack([
            constraint_features_4d,
            np.zeros((constraint_features_4d.shape[0], 6))
        ])
        
        variable_features_10d = np.hstack([
            variable_features,
            np.zeros((variable_features.shape[0], 4))
        ])
        
        # Try to extract edges from LP file
        edge_list = []
        edge_weights = []
        
        if os.path.exists(lp_file):
            try:
                import gurobipy as gp
                gp.setParam('OutputFlag', 0)
                env = gp.Env(empty=True)
                env.setParam('OutputFlag', 0)
                env.setParam('Threads', 1)
                env.start()
                
                lp_model = gp.read(lp_file, env=env)
                lp_model.setParam('OutputFlag', 0)
                lp_model.setParam('Threads', 1)
                
                for c_idx, constr in enumerate(lp_model.getConstrs()):
                    expr = lp_model.getRow(constr)
                    for i in range(expr.size()):
                        var = expr.getVar(i)
                        coeff = expr.getCoeff(i)
                        v_idx_str = var.VarName.split('[')[1].split(']')[0] if '[' in var.VarName else '0'
                        try:
                            v_idx = int(v_idx_str)
                            if 0 <= v_idx < num_vars:
                                edge_list.append([c_idx, v_idx + num_cons])
                                edge_weights.append(abs(coeff))
                        except:
                            pass
                
                print(f"edges={len(edge_list)} ", end="", flush=True)
            except Exception as e:
                print(f"(LP parse fail, ", end="", flush=True)
        
        # If no edges extracted, use sampled edges
        if len(edge_list) == 0:
            # Sample edges: each constraint connects to ~5-10% of variables randomly
            sample_rate = max(0.05, min(1.0, 10.0 / num_vars))
            for c_idx in range(num_cons):
                for v_idx in range(num_vars):
                    if np.random.rand() < sample_rate:
                        edge_list.append([c_idx, v_idx + num_cons])
                        edge_weights.append(1.0)
            print(f"sampled={len(edge_list)} ", end="", flush=True)
        
        # Create torch tensors
        if len(edge_list) > 0:
            edge_index = torch.LongTensor(edge_list).t().contiguous()
            edge_weight = torch.FloatTensor(edge_weights)
        else:
            edge_index = torch.LongTensor(2, 0)
            edge_weight = torch.FloatTensor([])
        
        # Build feature tensor
        constraint_tensor = torch.FloatTensor(constraint_features_10d)
        variable_tensor = torch.FloatTensor(variable_features_10d)
        feature_tensor = torch.cat([constraint_tensor, variable_tensor], dim=0)
        
        # Move to device
        feature_tensor = feature_tensor.to(device)
        edge_index = edge_index.to(device)
        edge_weight = edge_weight.to(device)
        
        # FORGE forward pass
        forge.eval()
        with torch.no_grad():
            h_list, logits, loss, indices, codebook_ = forge.forward(
                feature_tensor,
                num_cons,
                num_vars,
                edge_index,
                edge_weight
            )
            embedding = np.bincount(indices.cpu().numpy(), minlength=forge.codebook_size).astype(float)
        
        sat_embeddings_dict[instance_name] = embedding
        success_count += 1
        print(f"✓")
        
        # Cleanup
        del data, constraint_features, variable_features, constraint_features_10d, variable_features_10d
        del constraint_tensor, variable_tensor, feature_tensor, edge_index, edge_weight
        del h_list, logits, loss, indices, codebook_, embedding
        gc.collect()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
    except Exception as e:
        print(f"✗ {str(e)[:50]}")

print(f"\n✓ Successfully generated {success_count}/{len(instance_names)} embeddings")

# Save embeddings
if len(sat_embeddings_dict) > 0:
    print(f"Saving to {output_file}...")
    with open(output_file, 'wb') as f:
        pickle.dump(sat_embeddings_dict, f)
    print(f"✓ Saved {len(sat_embeddings_dict)} embeddings")


In [ ]:
#with open('../iclr_test_clusters_mip_to_embeddings_from_forge_pretrained.pkl', 'rb') as f:
with open('./iclr_test_clusters_mip_to_embeddings.pkl', 'rb') as f:
# with open('./satlib_mip_to_embeddings_from_sat_features.pkl', 'rb') as f:
     mip_to_embeddings = pickle.load(f)
     # mip_to_embeddings_dict = pickle.load(f)

with open('./iclr_test_clusters_mip_to_embeddings.pkl', 'rb') as f:
# with open('./satlib_mip_to_embeddings_from_sat_features.pkl', 'rb') as f:
     mip_to_embeddings_dict = pickle.load(f)

In [ ]:
mip_to_embeddings.keys()

# Visualize Embeddings

In [ ]:
color_vec = []
for key in mip_to_embeddings.keys():
    filename = key.split('/')[-1]  # Get just the filename
    
    # Check if it's SATLIB format
    if 'SATLIB_' in filename:
        # Extract category from format: SATLIB_<CATEGORY>-<instance>.mps
        parts = filename.replace('.mps', '').split('-')
        category = parts[0].replace('SATLIB_', '')
        color_vec.append(category)
    else:
        # Original MIP format: extract second underscore-separated part
        try:
            color_vec.append(filename.split('_')[1])
        except:
            color_vec.append(filename)

# Handle both numpy arrays (from SAT features) and objects with .instance_embedding (from FORGE pipeline)
embed_list = []
for key in mip_to_embeddings.keys():
    emb = mip_to_embeddings[key]
    if isinstance(emb, np.ndarray):
        embed_list.append(emb)
    else:
        embed_list.append(emb.instance_embedding)

embed_mat = np.array(embed_list)
# Large mip insances will have many nodes (ct/vars) so normalize by size
# Normalize row-wise, divide each row by its sum to get unit vectors
embed_mat = embed_mat / (embed_mat.sum(axis = 1, keepdims=True) + 1e-10)

In [ ]:
matrix_to_visualize = embed_mat

# Compute PCA to visualize in 2D
pca = pacmap.PaCMAP(n_components = 2, n_neighbors=10, MN_ratio=0.5, FP_ratio=2.0).fit_transform(matrix_to_visualize, init = 'pca')
# pca = PCA(n_components = 2).fit_transform(matrix_to_visualize)
pca = (pca - np.min(pca)) / np.ptp(pca)

# Compute an index dict to make it easier to plot different labels and colors
index_dict = {}
for idx, c in enumerate(color_vec):
    try:
        index_dict[c].append(idx)
    except:
        index_dict[c] = [idx]

cmap = plt.get_cmap('tab20')

# Plot Instances 
plt.figure(figsize = (5.5, 6.5), dpi = 300)
for idx, c in enumerate(sorted(index_dict)):
    plt.scatter(pca[index_dict[c], 0] + np.random.rand()/1000, pca[index_dict[c], 1] + np.random.rand()/1000, s = 10, label = c, color = cmap(idx))
# plt.legend(loc = 'right', ncols = 1, bbox_to_anchor = (1.45, .5))
plt.legend(loc = 'lower center', bbox_to_anchor = (0.5, 1.05), ncols = 2, fontsize = 'small')
plt.subplots_adjust(top = 0.85)

plt.show()

# Cluster and NMI

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score, adjusted_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix

def purity_score(y_true, y_pred):
    """
    Used for Clustering Accuracy
    """

    # Compute contingency matrix (also called confusion matrix)
    con_matrix = contingency_matrix(y_true, y_pred)

    # Return purity
    return np.sum(np.amax(con_matrix, axis=0)) / np.sum(con_matrix)


In [ ]:
num_clusters = len(set(color_vec))
res_nmi = {'dist' : [], }
res_acc = {'dist' : [], }

for _ in range(10):
    
    km_dist = KMeans(n_clusters = num_clusters).fit(embed_mat)
    km_type = {'dist' : km_dist}

    for t in ['dist']:
        true = color_vec
        pred = km_type[t].labels_
        
        acc = {c : 0 for c in range(num_clusters)}
        names = {c : None for c in range(num_clusters)}
        for c in range(num_clusters):
            indices = np.where(pred == c)[0]
            acc[c] = purity_score(np.array(true)[indices], pred[indices])
            names[c] = set(np.array(true)[indices])
        
        nmi_score = normalized_mutual_info_score(true, pred)
        mean_acc = np.mean(list(acc.values()))
        res_acc[t].append(mean_acc)
        res_nmi[t].append(nmi_score)

for t in res_nmi.keys():
    print ("Results for clustering based on {}:".format(t))
    print (f"NMI: {np.mean(res_nmi[t]):.3f} | ACC: {np.mean(res_acc[t]):.3f}")


# Inspect Raw Assigned Codes (Node-Level)


In [2]:
import os
import glob
import torch
import gurobipy as gp
import numpy as np
from forge.processor import _MIPUtils
from forge.embeddings import Forge

# Set up environment
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
gurobi_env = _MIPUtils.start_gurobi_env()

# Load forge model
print("Loading Forge model...")
forge_fresh = Forge(train_config_yaml="../forge/configs/train_config.yaml")
forge_fresh.load_state_dict(torch.load("../models/iclr_forge_pretrain_trained.pkl", map_location='cpu'))
forge_fresh.is_trained = True
forge_fresh.eval()
device = next(forge_fresh.parameters()).device
print(f"Model loaded on device: {device}\n")

# Load first test instance
instance_list_file = "../data/configs/iclr_test_clusters.txt"
with open(instance_list_file, 'r') as f:
    instances = [line.strip() for line in f if line.strip()]

test_instance_name = instances[0]
print(f"Testing instance: {test_instance_name}\n")

# Find and load MIP file
mip_files = glob.glob("../data/instances/**/*.lp", recursive=True)
matching_file = None
for f in mip_files:
    if test_instance_name.replace('.lp', '').replace('.mps', '') in f.replace('.lp', '').replace('.mps', ''):
        matching_file = f
        break

if matching_file:
    print(f"Found MIP file: {matching_file}\n")
    mip_model = gp.read(matching_file, env=gurobi_env)
    
    # Get embeddings
    with torch.no_grad():
        embeddings = forge_fresh._mip_model_to_embeddings(mip_model, instance_embedding_only=False)
    
    # Extract node embeddings and move to device for VQ processing
    eoc = embeddings.embedding_of_constraint.to(device)
    eov = embeddings.embedding_of_variable.to(device)
    all_embeddings = torch.cat([eoc, eov], dim=0)
    
    print(f"=== NODE-LEVEL EMBEDDINGS ===")
    print(f"Constraint embeddings shape: {eoc.shape}")
    print(f"Variable embeddings shape: {eov.shape}")
    print(f"Total node embeddings: {all_embeddings.shape}\n")
    
    # Get raw VQ indices from VQ layer
    forge_fresh.eval()
    with torch.no_grad():
        quantized, vq_indices, loss = forge_fresh.vq(all_embeddings.unsqueeze(0))
    
    raw_vq_indices = vq_indices.squeeze(0).cpu().numpy()
    
    print(f"=== RAW VQ INDICES ===")
    print(f"Total nodes: {len(raw_vq_indices)}")
    print(f"Unique codes: {len(np.unique(raw_vq_indices))}")
    print(f"Codes (sorted): {sorted(np.unique(raw_vq_indices).tolist())}")
    print(f"\n=== NODE-TYPE ANALYSIS ===")
    
    # Split by node type
    num_cons = eoc.shape[0]
    num_vars = eov.shape[0]
    constraint_indices = raw_vq_indices[:num_cons]
    variable_indices = raw_vq_indices[num_cons:]
    
    print(f"Total constraints: {num_cons}")
    print(f"Total variables: {num_vars}\n")
    
    # Analyze constraints
    constraint_codes = np.unique(constraint_indices)
    print(f"CONSTRAINTS - Unique codes: {sorted(constraint_codes.tolist())}")
    for code in sorted(constraint_codes):
        count = np.sum(constraint_indices == code)
        print(f"  Code {code}: {count} constraints ({100*count/num_cons:.1f}%)")
    
    # Analyze variables
    variable_codes = np.unique(variable_indices)
    print(f"\nVARIABLES - Unique codes: {sorted(variable_codes.tolist())}")
    for code in sorted(variable_codes):
        count = np.sum(variable_indices == code)
        print(f"  Code {code}: {count} variables ({100*count/num_vars:.1f}%)")
    
    # Global analysis
    print(f"\n=== GLOBAL CODE USAGE ===")
    for code in sorted(np.unique(raw_vq_indices)):
        total = np.sum(raw_vq_indices == code)
        cons = np.sum(constraint_indices == code)
        vars = np.sum(variable_indices == code)
        print(f"Code {code}: {total} total ({cons} constraints, {vars} variables)")
    
    print(f"\n✓ Analysis complete. Variables available for further analysis:")
    print(f"  raw_vq_indices, constraint_indices, variable_indices")
    print(f"  constraints, variables, mip_model, matching_file")

else:
    print(f"ERROR: Could not find MIP file for {test_instance_name}")

/opt/homebrew/Caskroom/miniforge/base/envs/forge_env/lib/python3.12/site-packages/gurobi_onboarder/app_cert/iconfig_constants.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Loading Forge model...
Model loaded on device: cpu

Testing instance: DMIPLIB_CA-easy_val_instance_9.lp

Found MIP file: ../data/instances/DMIPLIB_CA-easy_val_instance_945.lp

=== NODE-LEVEL EMBEDDINGS ===
Constraint embeddings shape: torch.Size([382, 1024])
Variable embeddings shape: torch.Size([1000, 1024])
Total node embeddings: torch.Size([1382, 1024])

=== RAW VQ INDICES ===
Total nodes: 1382
Unique codes: 6
Codes (sorted): [8, 41, 49, 52, 68, 94]

=== NODE-TYPE ANALYSIS ===
Total constraints: 382
Total variables: 1000

CONSTRAINTS - Unique codes: [68]
  Code 68: 382 constraints (100.0%)

VARIABLES - Unique codes: [8, 41, 49, 52, 94]
  Code 8: 190 variables (19.0%)
  Code 41: 6 variables (0.6%)
  Code 49: 7 variables (0.7%)
  Code 52: 567 variables (56.7%)
  Code 94: 230 variables (23.0%)

=== GLOBAL CODE USAGE ===
Code 8: 190 total (0 constraints, 190 variables)
Code 41: 6 total (0 constraints, 6 variables)
Code 49: 7 total (0 constraints, 7 variables)
Code 52: 567 total (0 const

In [3]:

# Scan all instances and find most active codes

import os
import glob
import torch
import gurobipy as gp
from forge.processor import _MIPUtils
import numpy as np
from collections import defaultdict

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

print(f"=== SCANNING ALL INSTANCES FOR CODE FREQUENCY ===\n")

# Load instance list
instance_list_file = "../data/configs/iclr_test_clusters.txt"
with open(instance_list_file, 'r') as f:
    instances = [line.strip() for line in f if line.strip()]

print(f"Total instances to check: {len(instances)}\n")

# Track code frequencies across all instances
code_instance_count = {}  # How many instances use each code
code_total_count = {}     # Total node count for each code

instance_count = 0

for idx, instance_name in enumerate(instances):
    # Find the MIP file
    mip_files = glob.glob("../data/instances/**/*.lp", recursive=True)
    matching_file = None
    for f in mip_files:
        if instance_name.replace('.lp', '').replace('.mps', '') in f.replace('.lp', '').replace('.mps', ''):
            matching_file = f
            break
    
    if matching_file:
        try:
            # Load model and get embeddings
            gurobi_env = _MIPUtils.start_gurobi_env()
            mip_model = gp.read(matching_file, env=gurobi_env)
            
            device = next(forge_fresh.parameters()).device
            embeddings = forge_fresh._mip_model_to_embeddings(mip_model, instance_embedding_only=False)
            
            eoc = embeddings.embedding_of_constraint.to(device)
            eov = embeddings.embedding_of_variable.to(device)
            all_embeddings = torch.cat([eoc, eov], dim=0)
            
            # Get VQ indices
            forge_fresh.eval()
            with torch.no_grad():
                quantized, vq_indices, loss = forge_fresh.vq(all_embeddings.unsqueeze(0))
            
            raw_vq_indices = vq_indices.squeeze(0).cpu().numpy()
            
            # Track code frequencies
            unique_codes, counts = np.unique(raw_vq_indices, return_counts=True)
            for code_idx, code_count in zip(unique_codes, counts):
                code = int(code_idx)
                count = int(code_count)
                
                if code not in code_instance_count:
                    code_instance_count[code] = 0
                    code_total_count[code] = 0
                
                code_instance_count[code] += 1
                code_total_count[code] += count
            
            instance_count += 1
                
        except Exception as e:
            pass
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(instances)} instances ({instance_count} success)...")

print(f"\n✓ Successfully scanned {instance_count} instances\n")
print(f"=== CODE FREQUENCY ANALYSIS ===\n")

# Sort codes by number of instances using them
if code_instance_count:
    sorted_codes = sorted(code_instance_count.items(), key=lambda x: x[1], reverse=True)
    
    print(f"{'Code':<6} {'Instances':<12} {'Total Nodes':<15} {'Avg Nodes/Instance':<20}")
    print("=" * 55)
    for code, num_instances in sorted_codes:
        total_nodes = code_total_count[code]
        avg_per_instance = total_nodes / num_instances if num_instances > 0 else 0
        print(f"{code:<6} {num_instances:<12} {total_nodes:<15} {avg_per_instance:<20.1f}")
    
    print(f"\n✓ Top codes for analysis: {[code for code, _ in sorted_codes[:5]]}")
    print(f"✓ For best results, use Code {sorted_codes[0][0]} (appears in {sorted_codes[0][1]} instances)")
else:
    print("No codes found - check model state")



=== SCANNING ALL INSTANCES FOR CODE FREQUENCY ===

Total instances to check: 1000

Processed 100/1000 instances (100 success)...
Processed 200/1000 instances (200 success)...
Processed 300/1000 instances (250 success)...
Processed 400/1000 instances (300 success)...
Processed 500/1000 instances (350 success)...
Processed 600/1000 instances (400 success)...
Processed 700/1000 instances (500 success)...
Processed 800/1000 instances (550 success)...
Processed 900/1000 instances (650 success)...
Processed 1000/1000 instances (750 success)...

✓ Successfully scanned 750 instances

=== CODE FREQUENCY ANALYSIS ===

Code   Instances    Total Nodes     Avg Nodes/Instance  
41     750          584538          779.4               
8      550          165175          300.3               
52     550          559977          1018.1              
94     550          332601          604.7               
49     546          54298           99.4                
91     350          2687501         7678.6

In [ ]:

# Generate prompts showing full optimization problem with highlighted Code 49 elements

import os
import glob
import json
import torch
import gurobipy as gp
from forge.processor import _MIPUtils
import numpy as np

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

target_code = 49
num_top_instances = 5

print(f"=== FINDING TOP {num_top_instances} INSTANCES FOR CODE {target_code} ===\n")

# Make sure model is loaded
if 'forge_fresh' not in locals():
    print("Loading Forge model...")
    forge_fresh = Forge(train_config_yaml="../forge/configs/train_config.yaml")
    forge_fresh.load_state_dict(torch.load("../models/iclr_forge_pretrain_trained.pkl", map_location='cpu'))
    forge_fresh.is_trained = True
    forge_fresh.eval()
    device = next(forge_fresh.parameters()).device
    print(f"Model loaded on device: {device}\n")
else:
    device = next(forge_fresh.parameters()).device
    print("Using existing Forge model\n")

instance_list_file = "../data/configs/iclr_test_clusters.txt"
with open(instance_list_file, 'r') as f:
    instances = [line.strip() for line in f if line.strip()]

# Scan instances to find where Code 49 is most active
instance_scores = []
processed = 0
errors = 0

for idx, instance_name in enumerate(instances):
    if processed >= 150:  # Sample 150 instances
        break
        
    mip_files = glob.glob("../data/instances/**/*.lp", recursive=True)
    matching_file = None
    for f in mip_files:
        if instance_name.replace('.lp', '').replace('.mps', '') in f.replace('.lp', '').replace('.mps', ''):
            matching_file = f
            break
    
    if matching_file:
        try:
            gurobi_env = _MIPUtils.start_gurobi_env()
            mip_model = gp.read(matching_file, env=gurobi_env)
            
            embeddings = forge_fresh._mip_model_to_embeddings(mip_model, instance_embedding_only=False)
            
            eoc = embeddings.embedding_of_constraint.to(device)
            eov = embeddings.embedding_of_variable.to(device)
            all_embeddings = torch.cat([eoc, eov], dim=0)
            
            forge_fresh.eval()
            with torch.no_grad():
                quantized, vq_indices, loss = forge_fresh.vq(all_embeddings.unsqueeze(0))
            
            raw_vq_indices = vq_indices.squeeze(0).cpu().numpy()
            
            count = np.sum(raw_vq_indices == target_code)
            if count > 0:
                instance_scores.append((instance_name, matching_file, count, len(raw_vq_indices), raw_vq_indices))
                print(f"✓ {instance_name}: {count} Code {target_code} nodes")
            
            processed += 1
        except Exception as e:
            errors += 1
            processed += 1
            if errors <= 3:  # Show first few errors
                print(f"✗ {instance_name}: {str(e)[:60]}")

print(f"\n✓ Processed {processed} instances, found {len(instance_scores)} with Code {target_code}\n")

if len(instance_scores) == 0:
    print(f"⚠ No instances found with Code {target_code}")
    print(f"Errors: {errors}")
else:
    instance_scores.sort(key=lambda x: x[2], reverse=True)
    
    print(f"=== TOP {min(num_top_instances, len(instance_scores))} INSTANCES ===\n")
    for rank, (name, path, count, total, _) in enumerate(instance_scores[:num_top_instances], 1):
        pct = 100 * count / total
        print(f"{rank}. {name}: {count}/{total} nodes ({pct:.1f}%)\n")

    # Generate detailed prompts with full problem context
    print(f"=== GENERATING DETAILED PROMPTS ===\n")

    examples_text = ""
    examples_json = []

    for rank, (instance_name, matching_file, count, total, indices) in enumerate(instance_scores[:num_top_instances], 1):
        print(f"Processing instance {rank}/{min(num_top_instances, len(instance_scores))}: {instance_name}...")
        
        try:
            gurobi_env = _MIPUtils.start_gurobi_env()
            mip_model = gp.read(matching_file, env=gurobi_env)
            
            constraints = mip_model.getConstrs()
            variables = mip_model.getVars()
            
            num_cons = len(constraints)
            num_vars = len(variables)
            
            constraint_indices = indices[:num_cons]
            variable_indices = indices[num_cons:]
            
            constraint_nodes = set(np.where(constraint_indices == target_code)[0])
            variable_nodes = set(np.where(variable_indices == target_code)[0])
            
            # Build full problem representation
            problem_text = f"\n{'='*80}\nExample {rank}: {instance_name}\n{'='*80}\n\n"
            problem_text += f"Optimization Problem:\n"
            problem_text += f"  Constraints: {num_cons}\n"
            problem_text += f"  Variables: {num_vars}\n"
            problem_text += f"  Code {target_code} assignments: {len(constraint_nodes)} constraints, {len(variable_nodes)} variables\n\n"
            
            # Show all constraints with highlights for code nodes
            problem_text += "Constraints:\n"
            for i, constr in enumerate(constraints):
                expr = mip_model.getRow(constr)
                
                if i in constraint_nodes:
                    constr_str = f"  << {expr} {constr.Sense} {constr.RHS} >>"
                else:
                    constr_str = f"  {expr} {constr.Sense} {constr.RHS}"
                
                problem_text += constr_str + "\n"
            
            problem_text += "\nVariables:\n"
            for i, var in enumerate(variables):
                var_name_full = f"{var.VarName} ∈ [{var.LB}, {var.UB}] ({var.VType})"
                
                if i in variable_nodes:
                    var_str = f"  << {var_name_full} >>"
                else:
                    var_str = f"  {var_name_full}"
                
                problem_text += var_str + "\n"
            
            examples_text += problem_text + "\n"
            
            examples_json.append({
                "example_number": rank,
                "instance_name": instance_name,
                "num_constraints": int(num_cons),
                "num_variables": int(num_vars),
                "code_49_constraints": int(len(constraint_nodes)),
                "code_49_variables": int(len(variable_nodes)),
                "full_problem": problem_text
            })
            print(f"  ✓ Generated\n")
        except Exception as e:
            print(f"  ✗ Error: {str(e)[:60]}\n")

    # Save files
    text_file = f"codeword_{target_code}_detailed_forgeinterp.txt"
    json_file = f"codeword_{target_code}_detailed_forgeinterp.json"
    
    with open(text_file, 'w') as f:
        f.write(examples_text)
    
    with open(json_file, 'w') as f:
        json.dump({
            "code": target_code,
            "num_instances": len(examples_json),
            "description": f"Top instances where Code {target_code} is most active. Highlighted elements (<<...>>) indicate constraints/variables assigned to Code {target_code}.",
            "examples": examples_json
        }, f, indent=2)

    print(f"✓ Created: {text_file}")
    print(f"✓ Created: {json_file}\n")
    
    print("=" * 80)
    print("PREVIEW (first 800 chars):")
    print("=" * 80)
    print(examples_text[:800])
    if len(examples_text) > 800:
        print("\n...(truncated)")

